# Incremental clickstream ingestion and scheduled processing

Process daily CSV batches in Databricks Free Edition using Auto Loader, a persistent checkpoint and Delta tables. Bronze appends newly discovered files; Silver and Gold are rebuilt from the accumulated, validated data.

## Verified result

The 2026-11-19 full-load Job run succeeded with **135 daily files, 165,474 clicks and 24,026 sessions**, covering 2008-04-01 through 2008-08-13. Gold contains 24,026 session rows, 135 daily rows and 540 date/entry-category rows. See [the validation record](../../reports/pipeline_validation.md) for the scheduled run, three-day rerun and full-load evidence.

This notebook contains the 12 code cells copied from the validated Databricks notebook, in their original order. Code is unchanged; headings were added for navigation. Outputs are cleared because the interactive notebook's older outputs do not represent the latest Job run. The validation record reports the observed Job outputs separately.

## Runtime and input contract

- Run in Databricks with Spark, `dbutils`, Unity Catalog and access to the existing `workspace.default.clickstream_data` volume. This is not a standalone local Python notebook.
- Upload semicolon-delimited files named `clickstream_YYYY-MM-DD.csv` to the `incoming` directory defined below. Preserve the original header and field order.
- Retain all previously ingested files unchanged: reconciliation reads all retained CSVs. Upload files between runs, not while a run is in progress.
- Reuse the existing checkpoint and Bronze table together. The `two_day_trial` path and `_trial` table names are retained from the original pilot and now contain all 135 days.
- Silver and Gold writes use `overwrite`; their named output tables are rebuilt on each run. Bronze uses `append` with the checkpoint.

## Job configuration

The deployed Job `clickstream_two_day_trial` runs this notebook as the `process_clickstream` task on serverless compute, with maximum concurrent runs set to 1. Its active schedule is daily at 14:50:55 in `Australia/Sydney` (`55 50 14 * * ?`). The schedule belongs to the Job and is not created by importing this notebook.

The source is a static historical dataset split into daily batches. Files were uploaded manually; this demonstrates incremental ingestion and scheduled processing, not a live daily source feed. GitHub Pages continues to use committed dashboard data and is not refreshed by this Job.


### 1. Create the input and checkpoint directories

In [ ]:
# Create folders for incoming files and ingestion progress.
base_path = "/Volumes/workspace/default/clickstream_data/two_day_trial"

incoming_path = f"{base_path}/incoming"
checkpoint_path = f"{base_path}/checkpoints/bronze"

dbutils.fs.mkdirs(incoming_path)
dbutils.fs.mkdirs(checkpoint_path)

print("Incoming path:", incoming_path)
print("Checkpoint path:", checkpoint_path)

### 2. Discover and validate daily filenames

In [ ]:
import re
from datetime import date

# Keep daily CSV files immutable and retain them for source reconciliation.
files = sorted(dbutils.fs.ls(incoming_path), key=lambda item: item.name)
assert files, "No input files were found."
input_paths = []
file_dates = {}
for file in files:
    match = re.fullmatch(r"clickstream_([0-9]{4}-[0-9]{2}-[0-9]{2})\.csv", file.name)
    assert match, f"Unexpected input file: {file.name}"
    batch_date = date.fromisoformat(match.group(1))
    assert file.size > 0, f"Empty file: {file.name}"
    input_paths.append(file.path)
    file_dates[file.name] = str(batch_date)
    print(f"File: {file.name} | Size: {file.size} bytes")

print("Daily input files:", len(input_paths))

### 3. Read the source CSVs and calculate reconciliation totals

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Match the original CSV column names and order.
source_schema = StructType([
    StructField("year", IntegerType(), True),
    StructField("month", IntegerType(), True),
    StructField("day", IntegerType(), True),
    StructField("order", IntegerType(), True),
    StructField("country", IntegerType(), True),
    StructField("session ID", IntegerType(), True),
    StructField("page 1 (main category)", IntegerType(), True),
    StructField("page 2 (clothing model)", StringType(), True),
    StructField("colour", IntegerType(), True),
    StructField("location", IntegerType(), True),
    StructField("model photography", IntegerType(), True),
    StructField("price", IntegerType(), True),
    StructField("price 2", IntegerType(), True),
    StructField("page", IntegerType(), True),
])

from pyspark.sql import functions as F

# Snapshot the discovered file list; do not upload files during a run.
source_csv = (
    spark.read.schema(source_schema)
    .option("header", True)
    .option("sep", ";")
    .option("mode", "FAILFAST")
    .option("enforceSchema", False)
    .csv(input_paths)
    .select("*", F.col("_metadata.file_path").alias("source_file"))
)

source_file_counts = {
    row["file_name"]: row["count"]
    for row in source_csv.withColumn(
        "file_name", F.regexp_extract("source_file", r"([^/]+)$", 1)
    ).groupBy("file_name").count().collect()
}
assert set(source_file_counts) == set(file_dates), "Missing or header-only CSV files."
expected_rows = sum(source_file_counts.values())
expected_sessions = source_csv.select("session ID").distinct().count()
expected_date_counts = {
    file_dates[name]: count for name, count in source_file_counts.items()
}

source_dates = source_csv.withColumn(
    "file_date", F.regexp_extract("source_file", r"clickstream_(\d{4}-\d{2}-\d{2})\.csv$", 1)
).withColumn(
    "record_date",
    F.expr("try_cast(concat(year, '-', lpad(month, 2, '0'), '-', lpad(day, 2, '0')) AS DATE)")
)
assert source_dates.filter(
    F.col("record_date").isNull()
    | (F.col("record_date").cast("string") != F.col("file_date"))
).limit(1).count() == 0, "CSV filename date does not match record date."

print("Source rows:", expected_rows)
print("Source sessions:", expected_sessions)
print("Source days:", len(expected_date_counts))
display(source_csv.limit(5))

### 4. Append new files to Bronze with Auto Loader

In [ ]:
from pyspark.sql import functions as F

bronze_table = "workspace.default.clickstream_bronze_trial"

# Discover CSV files using Auto Loader.
raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .schema(source_schema)
    .option("header", True)
    .option("sep", ";")
    .option("mode", "FAILFAST")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .load(incoming_path)
)

# Use Delta-compatible names and preserve ingestion details.
bronze_stream = raw_stream.select(
    F.col("year"),
    F.col("month"),
    F.col("day"),
    F.col("order").alias("click_order"),
    F.col("country"),
    F.col("session ID").alias("session_id"),
    F.col("page 1 (main category)").alias("main_category"),
    F.col("page 2 (clothing model)").alias("product_code"),
    F.col("colour").alias("color"),
    F.col("location"),
    F.col("model photography").alias("model_photography"),
    F.col("price"),
    F.col("price 2").alias("price_above_category_avg"),
    F.col("page"),
    F.col("_rescued_data"),
    F.col("_metadata.file_path").alias("source_file"),
    F.current_timestamp().alias("ingested_at"),
)

# Process currently available files, then stop.
query = (
    bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

print("Ingestion completed.")
print("Bronze table:", bronze_table)
print("Checkpoint path:", checkpoint_path)

### 5. Check Bronze totals, rescued data and checkpoint files

In [ ]:
# Read the persisted Bronze table.
bronze = spark.table(bronze_table)

row_count = bronze.count()
rescued_count = bronze.filter(
    F.col("_rescued_data").isNotNull()
).count()

print("Bronze rows:", row_count)
print("Rows with rescued data:", rescued_count)

display(
    bronze.groupBy("source_file")
    .count()
)

assert row_count == expected_rows, (
    f"Expected {expected_rows} Bronze rows from source CSVs, got {row_count}."
)
assert rescued_count == 0, "Inspect rows with rescued data."

# Confirm that Spark created checkpoint files.
checkpoint_files = dbutils.fs.ls(checkpoint_path)
print("Checkpoint entries:", len(checkpoint_files))

for file in checkpoint_files:
    print(file.name)

### 6. Compare Bronze records with the retained source files

In [ ]:
# Reconcile accumulated Bronze records with all retained source CSVs.
bronze = spark.table(bronze_table)
bronze_file_counts = {
    row["file_name"]: row["count"]
    for row in bronze.withColumn(
        "file_name", F.regexp_extract("source_file", r"([^/]+)$", 1)
    ).groupBy("file_name").count().collect()
}
assert bronze_file_counts == source_file_counts, "Per-file Bronze counts differ from source CSVs."

# Compare all business fields, including duplicate multiplicities.
business_columns = [
    "year", "month", "day", "click_order", "country", "session_id",
    "main_category", "product_code", "color", "location",
    "model_photography", "price", "price_above_category_avg", "page",
]
expected_bronze = source_csv.select(*[
    F.col(field.name).alias(target)
    for field, target in zip(source_schema.fields, business_columns)
])
actual_bronze = bronze.select(*business_columns)
assert expected_bronze.exceptAll(actual_bronze).limit(1).count() == 0, "Source records missing from Bronze."
assert actual_bronze.exceptAll(expected_bronze).limit(1).count() == 0, "Unexpected or repeated Bronze records."

session_count = bronze.select("session_id").distinct().count()
assert session_count == expected_sessions, "Bronze session count differs from source."
print("Bronze rows:", bronze.count())
print("Distinct sessions:", session_count)
display(bronze.groupBy("source_file").count().orderBy("source_file"))
print("Dynamic source-to-Bronze checks passed.")

### 7. Prepare the Silver records

In [ ]:
from pyspark.sql import functions as F

bronze_table = "workspace.default.clickstream_bronze_trial"
silver_table = "workspace.default.clickstream_silver_trial"

# Read the current Bronze data.
bronze = spark.table(bronze_table)

# Standardize product codes and build a date column.
silver_candidate = (
    bronze
    .withColumn("product_code", F.trim(F.col("product_code")))
    .withColumn(
        "event_date",
        F.expr("""
            try_cast(
                concat(
                    cast(year as string), '-',
                    lpad(cast(month as string), 2, '0'), '-',
                    lpad(cast(day as string), 2, '0')
                ) as date
            )
        """)
    )
)

display(
    silver_candidate.select(
        "event_date",
        "session_id",
        "click_order",
        "main_category",
        "product_code",
        "price",
    ).orderBy("session_id", "click_order").limit(5)
)

### 8. Check required fields, category codes and click keys

In [ ]:
# Require all original business fields and the derived date.
required_columns = [
    "year", "month", "day", "click_order", "country",
    "session_id", "main_category", "product_code", "color",
    "location", "model_photography", "price",
    "price_above_category_avg", "page", "event_date",
]

rules = {
    f"missing_{name}": F.col(name).isNull()
    for name in required_columns
}

# Apply the documented category ranges and basic value checks.
rules.update({
    "rescued_data": F.col("_rescued_data").isNotNull(),
    "blank_product_code": F.col("product_code") == "",
    "invalid_session_id": F.col("session_id") <= 0,
    "invalid_click_order": F.col("click_order") <= 0,
    "invalid_price": F.col("price") <= 0,
    "invalid_country": ~F.col("country").between(1, 47),
    "invalid_main_category": ~F.col("main_category").between(1, 4),
    "invalid_color": ~F.col("color").between(1, 14),
    "invalid_location": ~F.col("location").between(1, 6),
    "invalid_model_photography": ~F.col("model_photography").isin(1, 2),
    "invalid_price_flag": ~F.col("price_above_category_avg").isin(1, 2),
    "invalid_page": ~F.col("page").between(1, 5),
})

# Count rule violations in one aggregation.
quality_counts = silver_candidate.agg(*[
    F.count(F.when(condition, 1)).alias(name)
    for name, condition in rules.items()
]).first().asDict()

for name, count in quality_counts.items():
    print(f"{name}: {count}")

failed_rules = {
    name: count
    for name, count in quality_counts.items()
    if count > 0
}

assert not failed_rules, f"Data quality checks failed: {failed_rules}"

# A click must have a unique position within its session.
duplicate_keys = (
    silver_candidate
    .groupBy("session_id", "click_order")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_keys.count()
print("Duplicate session-order pairs:", duplicate_count)

assert duplicate_count == 0, "Inspect duplicate click keys before writing."

# Confirm that each session belongs to one date in this dataset.
cross_date_sessions = (
    silver_candidate
    .groupBy("session_id")
    .agg(F.countDistinct("event_date").alias("date_count"))
    .filter(F.col("date_count") > 1)
    .count()
)

print("Sessions spanning multiple dates:", cross_date_sessions)

assert cross_date_sessions == 0, "Inspect sessions spanning multiple dates."

print("Silver quality checks passed.")

### 9. Rebuild Silver and verify the saved records

In [ ]:
# Rebuild the trial table from the validated Bronze snapshot.
silver_ready = silver_candidate.drop("_rescued_data")

(
    silver_ready.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

# Read the saved table and verify against the source CSVs.
saved_silver = spark.table(silver_table)

summary = saved_silver.agg(
    F.count("*").alias("rows"),
    F.countDistinct("session_id").alias("sessions"),
    F.countDistinct("event_date").alias("days"),
).first()

print("Saved table:", silver_table)
print("Silver rows:", summary["rows"])
print("Silver sessions:", summary["sessions"])
print("Silver days:", summary["days"])

daily_counts = saved_silver.groupBy("event_date").count()

actual_counts = {
    str(row["event_date"]): row["count"]
    for row in daily_counts.collect()
}

assert actual_counts == expected_date_counts, "Silver daily counts differ from source CSVs."
assert summary["rows"] == expected_rows, "Silver row count differs from source."
assert summary["sessions"] == expected_sessions, "Silver session count differs from source."
assert summary["days"] == len(expected_date_counts), "Silver date count differs from source."

saved_records = saved_silver.select(*silver_ready.columns)
assert silver_ready.exceptAll(saved_records).limit(1).count() == 0, "Silver write lost records."
assert saved_records.exceptAll(silver_ready).limit(1).count() == 0, "Silver write added records."

daily_counts.orderBy("event_date").show()

print("Silver write and read-back checks passed.")

### 10. Build the Gold session table

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver = spark.table("workspace.default.clickstream_silver_trial")

# Identify the first recorded click in each session.
session_order = Window.partitionBy("session_id").orderBy("click_order")

first_clicks = (
    silver
    .withColumn("row_number", F.row_number().over(session_order))
    .filter(F.col("row_number") == 1)
    .select(
        "session_id",
        F.col("main_category").alias("entry_category"),
    )
)

# Aggregate all clicks within each session.
gold_sessions = (
    silver.groupBy("session_id")
    .agg(
        F.min("event_date").alias("event_date"),
        F.count("*").alias("click_count"),
        F.countDistinct("product_code").alias("distinct_product_count"),
        F.countDistinct("main_category").alias("distinct_category_count"),
    )
    .join(first_clicks, on="session_id", how="inner")
    .withColumn(
        "continued",
        (F.col("click_count") >= 2).cast("int"),
    )
    .withColumn(
        "repeat_product_views",
        F.col("click_count") - F.col("distinct_product_count"),
    )
)

# Reconcile the session summary with Silver.
source_rows = silver.count()
source_sessions = silver.select("session_id").distinct().count()

session_totals = gold_sessions.agg(
    F.count("*").alias("sessions"),
    F.countDistinct("session_id").alias("unique_sessions"),
    F.sum("click_count").alias("clicks"),
).first()

assert session_totals["sessions"] == source_sessions, "Session count mismatch."
assert session_totals["unique_sessions"] == source_sessions, "Duplicate session IDs."
assert session_totals["clicks"] == source_rows, "Click count mismatch."

print("Session summary rows:", session_totals["sessions"])
print("Summarized clicks:", session_totals["clicks"])

display(gold_sessions.orderBy("session_id").limit(5))

### 11. Build and reconcile daily and entry-category summaries

In [ ]:
# Build daily dashboard metrics from the session table.
gold_daily = (
    gold_sessions.groupBy("event_date")
    .agg(
        F.sum("click_count").alias("daily_clicks"),
        F.count("*").alias("daily_sessions"),
        F.sum("continued").alias("continued_sessions"),
    )
    .withColumn(
        "avg_clicks_per_session",
        F.col("daily_clicks") / F.col("daily_sessions"),
    )
)

# Keep the numerator and denominator for continuation rates.
gold_entry_daily = (
    gold_sessions.groupBy("event_date", "entry_category")
    .agg(
        F.count("*").alias("starting_sessions"),
        F.sum("continued").alias("continued_sessions"),
        F.sum("click_count").alias("clicks"),
    )
    .withColumn(
        "continuation_rate",
        F.col("continued_sessions") / F.col("starting_sessions"),
    )
)

# Validate each day's click and session totals against Silver.
expected_daily = {
    str(row["event_date"]): (row["clicks"], row["sessions"])
    for row in silver.groupBy("event_date").agg(
        F.count("*").alias("clicks"),
        F.countDistinct("session_id").alias("sessions"),
    ).collect()
}

actual_daily = {
    str(row["event_date"]): (row["daily_clicks"], row["daily_sessions"])
    for row in gold_daily.collect()
}

assert actual_daily == expected_daily, "Daily metrics do not match Silver."

# Every session belongs to exactly one starting category.
entry_totals = gold_entry_daily.agg(
    F.sum("starting_sessions").alias("sessions"),
    F.sum("clicks").alias("clicks"),
).first()

assert entry_totals["sessions"] == source_sessions, "Starting-session mismatch."
assert entry_totals["clicks"] == source_rows, "Category-summary click mismatch."

invalid_rates = gold_entry_daily.filter(
    F.col("continuation_rate").isNull()
    | ~F.col("continuation_rate").between(0, 1)
).count()

assert invalid_rates == 0, "Invalid continuation rates."

display(gold_daily.orderBy("event_date"))
display(gold_entry_daily.orderBy("event_date", "entry_category"))

print("Gold aggregation checks passed.")

### 12. Write Gold tables and compare saved records

In [ ]:
# Rebuild each trial table from the validated Silver data.
gold_tables = {
    "workspace.default.clickstream_sessions_trial": gold_sessions,
    "workspace.default.clickstream_daily_trial": gold_daily,
    "workspace.default.clickstream_entry_daily_trial": gold_entry_daily,
}

for table_name, dataframe in gold_tables.items():
    dataframe.write.format("delta").mode("overwrite").saveAsTable(table_name)

    # Compare saved records with the expected result in both directions.
    saved = spark.table(table_name).select(*dataframe.columns)

    missing_rows = dataframe.exceptAll(saved).limit(1).count()
    unexpected_rows = saved.exceptAll(dataframe).limit(1).count()

    assert missing_rows == 0, f"Missing records in {table_name}"
    assert unexpected_rows == 0, f"Unexpected records in {table_name}"

    print(f"Verified table: {table_name} | Rows: {saved.count()}")

print("All Gold tables passed write and read-back checks.")